# Naturalezza: pilota Gemma 4 31B tramite Google AI Studio

Questo notebook usa il branch **feature/gemma-naturalness-judge** della repository Humanization. Il giudice fissato è **gemma-4-31b-it**, tramite l'API nativa Google. La presenza nel catalogo del proprio account e il supporto della configurazione sono verificati durante l'esecuzione; non si sceglie automaticamente un altro modello o una modalità diversa.

Il riferimento Qwen contiene **61 originali completi** (17 Essay, 20 Reuters, 24 WritingPrompts). Ogni originale ha le stesse quattro riscritture, quattro ordini e tre ripetizioni: **732 richieste nel piano completo**. I due pannelli Qwen parziali restano esclusi. Il piano non cambia tra pilota e studio.

Si eseguono prima soltanto **10 pannelli, 120 richieste**, per verificare trasporto, quote e token di pensiero. **thinkingLevel=minimal non viene considerato sinonimo di thinking disattivato**: il controllo usa le risposte e i metadati effettivi, distinguendo zero esplicito e zero inferito dal bilancio dei token. Pensieri positivi o evidenza insufficiente impediscono il proseguimento.

Servono **CPU, Internet e il Secret GOOGLE_API_KEY** abilitato nelle impostazioni Kaggle. Nessun modello viene scaricato e nessuna valutazione viene presentata come annotazione umana. La concordanza fra giudici va misurata, non presupposta.


## 1. Configurazione

Al primo avvio lasciare `STAGE = "pilot"`, impostare `RUN_API = True` e tenere `RESUME_ZIP = ""`. Il nuovo output è distinto da Qwen e dal tentativo Llama.

Il limite di **4.000 POST** include ogni tentativo, compresi 429, 503, retry e riprese. Dopo gli errori temporanei si attende progressivamente 20, 40, 60 e 60 secondi, entro il limite configurato per singola richiesta. Nessuna risposta con formato non valido viene ripetuta fino a ottenere un voto desiderato.

Per riprendere dopo una sessione, caricare il più recente ZIP **Gemma Google** negli Input e usare `RESUME_ZIP = "auto"`, anche se è già estratto. Per passare al resto dello studio, soltanto dopo il superamento del controllo pilota, usare `STAGE = "full"` nella stessa cartella: piano, risposte e tentativi rimangono gli stessi.

Per correggere un checkpoint della prima versione, impostare **`REPARSE_SAVED_RESPONSES = True`**. Il notebook carica la revisione aggiornata del codice dal branch, rilegge le risposte grezze senza nuove richieste e conserva una copia dell'originale. Piano, prompt, voti restituiti e contatore POST rimangono gli stessi. La correzione accetta soltanto parti di pensiero vuote con zero token verificato e un unico involucro Markdown JSON; non ricostruisce voti o testo mancante.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, stat, subprocess, sys, tempfile, time, zipfile

RUN_API = False
REPARSE_SAVED_RESPONSES = False   # True per recuperare risposte salvate con il vecchio parser
STAGE = "pilot"                  # "full" soltanto dopo il superamento del pilota
RESUME_ZIP = ""                   # Prima volta vuoto; poi "auto" per il checkpoint Google
RUN_SUFFIX = "v1"
PROVIDER = "google"
JUDGE_MODEL = "gemma-4-31b-it"
THINKING_LEVEL = "minimal"
REPO_REF = "feature/gemma-naturalness-judge"
MAX_POST_ATTEMPTS = 4000
MAX_RETRIES = 4
MIN_REQUEST_INTERVAL = 8.0
BOOTSTRAP = 2000
SEED = 42
INPUT_TOKENS_PER_MINUTE = 15000   # Quota indicata dall'utente: controllarla nel proprio account
REFERENCE_SHA256 = "c834bf2d03eb3735c93fe105d7a937983f0c44a468784b1969ec997d46f6dee4"

assert STAGE in {"pilot", "full"}
assert PROVIDER == "google" and JUDGE_MODEL == "gemma-4-31b-it" and THINKING_LEVEL == "minimal"
assert Path("/kaggle/working").is_dir()
RUN = Path("/kaggle/working") / f"humanization_naturalness_panel_gemma_google_{RUN_SUFFIX}"
REPO = Path("/kaggle/working/.humanization_naturalness_google_code/Humanization")
RUN.mkdir(parents=True, exist_ok=True)
MPL_CACHE = Path("/kaggle/working/.humanization_naturalness_google_cache/matplotlib")
MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPL_CACHE)
# La fase è un limite operativo: cambiarla non modifica il piano congelato.
frozen_settings = {"mode": "google_cross_family_complete_panels", "provider": PROVIDER,
                   "model": JUDGE_MODEL, "thinking_level": THINKING_LEVEL,
                   "reference_checkpoint_sha256": REFERENCE_SHA256, "seed": SEED}
print("Provider:", PROVIDER, "| modello:", JUDGE_MODEL, "| livello richiesto:", THINKING_LEVEL)
print("Fase:", STAGE, "| API attive:", RUN_API, "| cartella:", RUN)


## 2. Ripresa del nuovo run Google

Alla prima esecuzione non ci sono giudizi Gemma da recuperare. Una ripresa controlla provider, modello, livello richiesto, riferimento e commit del codice. Non usare ZIP Qwen o Llama come ripresa Google. Gli Input vengono soltanto letti; gli avanzamenti locali hanno precedenza nella ricerca automatica.


In [ ]:
def matching_checkpoint_directories(root, settings):
    matches = []
    for path in root.rglob("runtime_info.json"):
        folder = path.parent
        if not all((folder / name).is_file() for name in
                   ["judge_plan.json", "judgments.jsonl", "api_attempts.jsonl"]):
            continue
        try:
            metadata = json.loads(path.read_text())
        except (OSError, ValueError):
            continue
        if metadata.get("settings") == settings:
            matches.append(folder)
    return sorted(set(matches))

resume_source = RESUME_ZIP
if resume_source == "auto":
    local_files = [RUN / name for name in
                   ["runtime_info.json", "judge_plan.json", "judgments.jsonl", "api_attempts.jsonl"]]
    if all(path.is_file() for path in local_files) and (RUN / "judgments.jsonl").stat().st_size:
        resume_source = ""
        print("Riprendo i file già presenti nella cartella del run.")
    else:
        matches = matching_checkpoint_directories(Path("/kaggle/input"), frozen_settings)
        assert len(matches) == 1, (
            "Serve un unico checkpoint estratto compatibile con modello e riferimento. "
            f"Cartelle trovate: {[str(path) for path in matches]}. "
            "Indicare esplicitamente il percorso corretto in RESUME_ZIP se necessario."
        )
        resume_source = str(matches[0])

if resume_source:
    resume_path = Path(resume_source)
    assert resume_path.exists(), f"Checkpoint da riprendere assente: {resume_path}"
    if resume_path.is_dir():
        if not (resume_path / "runtime_info.json").is_file():
            matches = matching_checkpoint_directories(resume_path, frozen_settings)
            assert len(matches) == 1, "Indicare la cartella che contiene i file del checkpoint."
            resume_path = matches[0]
        metadata = json.loads((resume_path / "runtime_info.json").read_text())
        assert metadata["settings"] == frozen_settings, "Parametri diversi dal checkpoint: controllare modello e riferimento."
        assert metadata["requested_ref"] == REPO_REF, "REPO_REF diverso dal checkpoint."
        assert all((resume_path / name).is_file() for name in
                   ["judge_plan.json", "judgments.jsonl", "api_attempts.jsonl"]), "Checkpoint incompleto."
        files = [path for path in resume_path.rglob("*") if path.is_file()
                 and path.suffix.lower() in {".json", ".jsonl", ".csv", ".png"}]
        assert sum(path.stat().st_size for path in files) <= 512 * 2**20, "Checkpoint troppo grande."
        resume_cache = RUN.parent / ".humanization_naturalness_google_cache"
        resume_cache.mkdir(parents=True, exist_ok=True)
        repacked = resume_cache / "checkpoint_resume.zip"
        with zipfile.ZipFile(repacked, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(files):
                archive.write(path, path.relative_to(resume_path).as_posix())
        print("Checkpoint letto dalla cartella Input:", resume_path)
        resume_path = repacked
    assert resume_path.is_file(), f"Checkpoint ZIP non trovato: {resume_path}"
    with zipfile.ZipFile(resume_path) as archive:
        assert "runtime_info.json" in archive.namelist(), "Lo ZIP non contiene runtime_info.json alla radice."
        metadata = json.loads(archive.read("runtime_info.json"))
        assert metadata["settings"] == frozen_settings, "Parametri diversi dal checkpoint: controllare modello e riferimento."
        assert metadata["requested_ref"] == REPO_REF, "REPO_REF diverso dal checkpoint."
        assert sum(item.file_size for item in archive.infolist()) <= 512 * 2**20, "ZIP di risultati troppo grande."
        for item in archive.infolist():
            assert not any(part in {".", ".."} for part in item.filename.split("/")), "Componenti ZIP non ammesse."
            relative = Path(item.filename)
            assert not relative.is_absolute() and ".." not in relative.parts and "\\" not in item.filename, "Percorso ZIP non valido."
            assert relative.parts and relative.parts[0] != ".", "Percorso ZIP vuoto."
            assert not stat.S_ISLNK(item.external_attr >> 16), "I collegamenti simbolici non sono ammessi."
            target = (RUN / relative).resolve()
            assert target == RUN.resolve() or RUN.resolve() in target.parents, "Percorso esterno alla cartella del run."
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            assert target.suffix.lower() in {".json", ".jsonl", ".csv", ".png"}, f"File inatteso nello ZIP: {relative}"
            content = archive.read(item)
            if target.exists():
                assert target.read_bytes() == content, f"File diverso già presente: {relative}. Usare una nuova cartella."
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(content)
    print("Risultati precedenti ripristinati senza sovrascritture.")

runtime_path = RUN / "runtime_info.json"
saved_runtime = json.loads(runtime_path.read_text()) if runtime_path.exists() else None
if saved_runtime:
    assert saved_runtime["settings"] == frozen_settings, "Parametri diversi dal run precedente: scegliere un nuovo RUN_SUFFIX."
    assert saved_runtime["requested_ref"] == REPO_REF, "REPO_REF diverso: ripristinare il riferimento precedente o cambiare cartella."
    print("Commit congelato per la ripresa:", saved_runtime["code_commit"])
    for name, label in [("judgments.jsonl", "Risposte già salvate"), ("api_attempts.jsonl", "Tentativi già registrati")]:
        path = RUN / name
        count = sum(bool(line.strip()) for line in path.read_text().splitlines()) if path.exists() else 0
        print(label + ":", count)
else:
    print("Nuovo run: nessuna risposta precedente da recuperare.")


## 3. Codice dal branch e librerie essenziali

Si scarica il branch indicato, in una cartella distinta. Non si modificano il vecchio notebook o i risultati Qwen. Bastano requests, numpy, pandas e matplotlib; nelle riprese viene riutilizzato il commit del primo run Google.


In [ ]:
repository_url = "https://github.com/FrancescoM00/Humanization.git"
REPO.parent.mkdir(parents=True, exist_ok=True)
fresh_clone = not REPO.exists()
if fresh_clone:
    subprocess.run(["git", "clone", "--depth", "1", repository_url, str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), f"Cartella esistente non riconosciuta come repository: {REPO}"
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True)
    assert not dirty.strip(), "La copia del codice ha modifiche locali: conservarle e scegliere un'altra cartella REPO."
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip()
    assert remote.removesuffix(".git") == repository_url.removesuffix(".git"), "Repository diversa da quella prevista."

selected_ref = (saved_runtime["code_commit"]
                if saved_runtime and not REPARSE_SAVED_RESPONSES else REPO_REF)
if not (fresh_clone and selected_ref == "main"):
    subprocess.run(["git", "fetch", "--depth", "1", "origin", selected_ref], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert (REPO / "scripts/compare_naturalness_google.py").is_file(), "Il riferimento scelto non contiene il nuovo script."
print("Repository:", REPO)
print("Commit utilizzato:", CODE_COMMIT)

minimal_packages = {
    "requests": "requests>=2.32,<3", "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2.2,<4", "matplotlib": "matplotlib>=3.8,<4",
}
missing = [requirement for module, requirement in minimal_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
import pandas as pd
from IPython.display import FileLink, Image, display
from importlib.metadata import version
LIBRARY_VERSIONS = {name: version(name) for name in minimal_packages}
print("Librerie:", LIBRARY_VERSIONS)


## 4. Riferimento congelato e ritmo delle richieste

I 61 originali e le riscritture sono inclusi nella repository. Il campione viene filtrato esclusivamente per completezza Qwen, senza selezionare in base ai voti. Le istruzioni di sistema e il messaggio utente restano identici: l'adattatore Google usa `systemInstruction` e `contents` e congela il payload realmente inviato. Non si assume il supporto a un vincolo JSON nativo: restano le stesse istruzioni e il parser rigoroso.

Il ritmo viene confrontato con i token di input effettivamente registrati da Qwen. Nel checkpoint sono circa **917 token di input per risposta**, non 4.000: a 15.000 token/minuto, con margine del 20%, risultano circa 4,4 secondi. Il valore iniziale di **8 secondi** è più prudente. Tokenizzatori e quote differiscono: il pilota registra l'uso Google e ogni 429.


In [ ]:
REFERENCE = REPO / "exports/naturalness_judge_comparison/qwen_reference_20261118.zip"
assert REFERENCE.is_file()
assert hashlib.sha256(REFERENCE.read_bytes()).hexdigest() == REFERENCE_SHA256
with zipfile.ZipFile(REFERENCE) as archive:
    qwen_manifest = json.loads(archive.read("judge_run.json"))
    qwen_rows = [line for line in archive.read("judgments.jsonl").splitlines() if line.strip()]
mean_input_tokens = qwen_manifest["token_usage"]["prompt_tokens"] / len(qwen_rows)
recommended_seconds = mean_input_tokens * 60 / INPUT_TOKENS_PER_MINUTE * 1.2
print("Media input Qwen:", round(mean_input_tokens, 1), "token; pausa stimata con margine:", round(recommended_seconds, 2))
print("Pausa scelta:", MIN_REQUEST_INTERVAL, "secondi. Le quote Google vanno verificate nell'account.")
runtime = {"settings": frozen_settings, "requested_ref": REPO_REF, "code_commit": CODE_COMMIT,
           "python": sys.version.split()[0], "library_versions": LIBRARY_VERSIONS}
runtime_upgrade_pending = bool(saved_runtime and saved_runtime["code_commit"] != CODE_COMMIT)
if runtime_upgrade_pending:
    assert REPARSE_SAVED_RESPONSES, "L'aggiornamento del parser richiede una rilettura esplicita."
    print("Aggiornamento del parser richiesto; il nuovo commit sarà registrato dopo la rilettura offline.")
elif not saved_runtime:
    runtime_path.write_text(json.dumps(runtime, ensure_ascii=False, indent=2) + "\n")

SCRIPT = REPO / "scripts/compare_naturalness_google.py"
def comparison_command(execute=False):
    command = [sys.executable, str(SCRIPT), "--reference", str(REFERENCE), "--output", str(RUN),
               "--provider", PROVIDER, "--model", JUDGE_MODEL, "--stage", STAGE,
               "--max-requests", str(MAX_POST_ATTEMPTS), "--max-retries", str(MAX_RETRIES),
               "--min-request-interval", str(MIN_REQUEST_INTERVAL), "--bootstrap", str(BOOTSTRAP),
               "--seed", str(SEED)]
    if REPARSE_SAVED_RESPONSES:
        command.append("--reparse-saved")
    if execute:
        command.append("--execute")
    return command

def load_json(name):
    path = RUN / name
    return json.loads(path.read_text()) if path.is_file() else {}

def read_rows(name):
    path = RUN / name
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.is_file() else []


## 5. Preparazione offline e controllo del piano

Devono comparire **61 originali, 732 richieste totali e un pilota di 10 originali / 120 richieste**. La preparazione non legge chiavi e non effettua inferenza. Il riferimento Qwen rimane separato dal registro Google.


In [ ]:
offline_env = dict(os.environ)
for name in ["GOOGLE_API_KEY", "GROQ_API_KEY"]:
    offline_env.pop(name, None)
subprocess.run(comparison_command(False), cwd=REPO, env=offline_env, check=True)
if runtime_upgrade_pending:
    # Si registra l'aggiornamento soltanto dopo il recupero offline riuscito.
    history = RUN / "reparse_v2"
    history.mkdir(parents=True, exist_ok=True)
    before = history / "runtime_before.json"
    if not before.exists():
        before.write_bytes(runtime_path.read_bytes())
    upgrade = {"from_commit": saved_runtime["code_commit"], "to_commit": CODE_COMMIT,
               "reason": "response format parser v2; saved raw responses reparsed offline",
               "api_requests_for_reparse": 0}
    with (RUN / "runtime_code_upgrades.jsonl").open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(upgrade, ensure_ascii=False) + "\n")
    runtime_path.write_text(json.dumps(runtime, ensure_ascii=False, indent=2) + "\n")
    saved_runtime = runtime
    runtime_upgrade_pending = False
    print("Commit aggiornato dopo la rilettura; runtime precedente conservato.")
setup = load_json("comparison_setup.json")
plan = load_json("judge_plan.json")
assert setup["n_complete_reference_sources"] == 61
assert plan["expected_completion_requests"] == 732
assert setup["n_pilot_sources"] == 10 and setup["expected_pilot_requests"] == 120
print("Piano completo:", setup["n_complete_reference_sources"], "originali /", plan["expected_completion_requests"], "richieste")
print("Pilota:", setup["n_pilot_sources"], "originali /", setup["expected_pilot_requests"], "richieste")
print("Pannelli Qwen parziali esclusi:", setup["n_excluded_partial_sources"])
print("Risposte Google già salvate:", len(read_rows("judgments.jsonl")))
print("Risposte Google valide:", load_json("judge_run.json").get("completed_valid_requests"))
print("Tentativi POST conservati:", load_json("judge_run.json").get("generation_post_attempts"))


## 6. Eseguire il pilota o riprenderlo

Il Secret **GOOGLE_API_KEY** è letto soltanto con `RUN_API = True`. Il catalogo Google è paginato; il modello deve essere presente e supportare `generateContent`. Non si cambiano automaticamente modello, istruzioni o livello di pensiero dopo un errore.

Le risposte grezze, la versione effettiva del modello e i conteggi restano salvati anche quando un voto è scartato. Il pilota verifica 120 risposte registrate, token di pensiero verificati a zero, nessuna parte di pensiero non vuota, versione coerente e meno del 5% di risposte non valide. Se un metadato manca, zero può essere inferito **soltanto** dal bilancio completo dei token; questa evidenza è etichettata separatamente da uno zero esplicito.

Un 503 è trattato come errore temporaneo del servizio e ritentato entro i limiti; 429 e relative attese restano nel registro. Scaricare sempre lo ZIP, anche dopo un arresto. In fase `pilot` non vengono giudicati i restanti 51 originali.

Le parti `thought=true` vuote vengono contate come marcatori vuoti e non come testo di ragionamento. Sono ignorate solo se il controllo dei token conferma zero. Un unico blocco Markdown che racchiude l'intero JSON viene scartato come involucro; il contenuto JSON resta soggetto agli stessi controlli di schema, ID, tipi e intervallo dei voti. La risposta originale resta disponibile integralmente.


In [ ]:
stage_ids = {item["request_id"] for item in plan["requests"]
             if STAGE == "full" or item["source_id"] in plan["pilot_source_ids"]}
answered_ids = {row["request_id"] for row in read_rows("judgments.jsonl")}
pending_ids = stage_ids - answered_ids
full_blocked = STAGE == "full" and not load_json("pilot_check.json").get("passed")
if RUN_API and pending_ids and not full_blocked:
    from kaggle_secrets import UserSecretsClient
    api_secret = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    assert isinstance(api_secret, str) and api_secret.strip(), "Secret GOOGLE_API_KEY non disponibile."
    api_env = dict(os.environ)
    api_env["GOOGLE_API_KEY"] = api_secret.strip()
    started = time.monotonic()
    try:
        outcome = subprocess.run(comparison_command(True), cwd=REPO, env=api_env, check=False)
    finally:
        api_env.pop("GOOGLE_API_KEY", None)
        del api_secret
    print("Durata di questa esecuzione, minuti:", round((time.monotonic()-started)/60, 2))
    if outcome.returncode:
        print("Il comando si è fermato: controllare l'errore e conservare comunque lo ZIP.")
elif not RUN_API:
    print("RUN_API=False: nessuna chiamata e nessuna chiave letta.")
elif full_blocked:
    print("Studio completo non avviato: il controllo pilota non è superato. Nessuna chiave letta.")
else:
    print("Questa fase ha già tutte le risposte registrate: nessuna nuova chiamata e nessuna chiave letta.")
manifest = load_json("judge_run.json")
for name in ["status", "blocked_reason", "completed_valid_requests", "expected_completion_requests",
             "n_complete_source_panels", "generation_post_attempts", "token_usage"]:
    print(name + ":", manifest.get(name))
print("Controllo pilota:", load_json("pilot_check.json"))


## 7. Leggere i controlli prima di proseguire

`pilot_check.json` riassume completamento, risposte scartate, token di pensiero e problemi di quota. `judgments.jsonl` conserva risposte grezze e prove per ciascuna chiamata; `api_attempts.jsonl` conserva ogni tentativo, compresi quelli falliti. Gli eventuali errori HTTP sono tracciati separatamente.

`comparison/` confronta Qwen e Gemma soltanto sugli stessi originali completi per entrambi. I voti parziali restano disponibili ma esclusi dalle medie appaiate. Un giudice di un'altra famiglia è un controllo di sensibilità; non garantisce neutralità o validazione umana.

Dopo il pilota, conservare lo ZIP e controllare che il controllo sia superato. Solo allora si può impostare `STAGE = "full"` e rieseguire: il controllo viene verificato anche dal runner, senza azzerare i tentativi o ripetere le risposte. Se compaiono token di pensiero o l'evidenza è insufficiente, fermarsi e discutere il vincolo con il professore.


In [ ]:
print("Controllo pilota:", load_json("pilot_check.json"))
subprocess.run(comparison_command(False), cwd=REPO, env=offline_env, check=True)
summary = load_json("comparison/summary.json")
print("Originali di riferimento:", summary.get("n_complete_reference"))
print("Originali completi per entrambi:", summary.get("n_common_complete_sources"))
print("Stato confronto:", summary.get("status"))
comparison_folder = RUN / "comparison"
for name in ["system_summary_by_judge.csv", "humanizer_contrast_shifts.csv",
             "system_shifts.csv", "ordinal_agreement.csv", "position_effects_by_judge.csv"]:
    path = comparison_folder / name
    if path.is_file():
        try:
            frame = pd.read_csv(path)
        except pd.errors.EmptyDataError:
            continue
        print(name, "— righe:", len(frame))
        display(frame.head(10))
for figure in sorted(comparison_folder.glob("*.png")):
    display(Image(filename=str(figure)))
if not summary.get("n_common_complete_sources"):
    print("Il confronto sarà disponibile dopo le prime dodici risposte Gemma valide per un originale.")


## 8. Scaricare il checkpoint

Lo ZIP include piano completo, riferimento, pilota, risposte native disponibili, metadati, tabelle e grafici. Non contiene credenziali, ambiente virtuale o pesi. Salvarlo anche se lo stato è parziale.

In una nuova sessione usare lo ZIP **Gemma Google** più recente negli Input e `RESUME_ZIP = "auto"`. La cartella del run resta la stessa anche passando da `pilot` a `full`: non cambiare `RUN_SUFFIX` per quel passaggio. Non si riprendono mai checkpoint Qwen o Llama come se fossero Google.


In [ ]:
archive_path = Path(shutil.make_archive(str(RUN), "zip", root_dir=RUN))
with zipfile.ZipFile(archive_path) as archive:
    assert not any(Path(name).name == ".env" for name in archive.namelist())
    assert not any(name.startswith((".git/", ".venv/")) for name in archive.namelist())
print("Archivio da scaricare:", archive_path)
print("Dimensione MiB:", round(archive_path.stat().st_size / 2**20, 2))
display(FileLink(str(archive_path)))
